In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Do I get money for landing on the free space?",
    "Never put cash in the centre of the board; you don't get a bonus for landing on Free Parking!",
    "Each player is dealt two private cards known as hole cards.",
]
vectors = model.encode(sentences, normalize_embeddings=True)
print(vectors.shape)
print(vectors @ vectors.T)

/Users/yasasvikotagiri/Desktop/Game/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 15:13:03.309000 45462 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18554.88it/s]


(3, 384)
[[1.0000001e+00 5.4870176e-01 7.6506846e-04]
 [5.4870176e-01 1.0000000e+00 1.7156750e-01]
 [7.6506846e-04 1.7156750e-01 1.0000000e+00]]


In [2]:
import json
import numpy as np

with open("../data/processed/chunks.json") as f:
    chunks = json.load(f)

chunk_texts = [c["text"] for c in chunks]
chunk_vectors = model.encode(chunk_texts, normalize_embeddings=True)
print(chunk_vectors.shape)   # should be (113, 384)

def search(question, k=5):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q             # one score per chunk
    top = np.argsort(scores)[::-1][:k]     # indices of the k highest scores
    for i in top:
        print(f"{scores[i]:.3f}  {chunks[i]['id']}  |  {chunks[i]['text'][:100]!r}")

(113, 384)


In [4]:
search("Do you collect money for landing on Free Parking?")


0.445  monopoly-p2-c5  |  '•\t You can double the rent for those streets! •\t You may build Houses and Hotels and charge even mor'
0.368  monopoly-p2-c0  |  'IN\n\nACTION SPACES\n\n3\n\nShuffle the Community Chest cards, and place them face down here.\n\n1\n\nCOLLECT '
0.314  monopoly-p2-c3  |  'Go To Jail Move your piece to the In Jail space immediately! Do not collect M200 for passing GO. You'
0.308  catan-p3-c2  |  'Build 2 roads at no cost.\n\ngive you all their resource\n\nfrom the supply.\n\nthem) to win the game.\n\nca'
0.299  catan-p6-c3  |  '2. ACTIVATE THE ROBBER You must move the robber to a new hex. Steal 1 random resource card from a pl'


In [5]:
search("How many private hole cards is each player dealt?")

0.654  texas_holdem-p2-c1  |  "In Hold'em, each player is dealt two private cards (known as ‘hole cards') that belong to them alone"
0.605  rummy-p4-c1  |  'The Deal: The number of cards dealt depends on the number of players. If there are 2 players then th'
0.584  poker-p6-c1  |  '2, 3 or 4 players: Stud Poker in any form. Usually, with so few players, only the very experienced p'
0.533  poker-p4-c2  |  'Poker is almost always played with poker chips. For a game with seven or more players, there should '
0.531  poker-p1-c3  |  'In clubs, it is customary to change cards often and to permit any player to call for new cards whene'


In [6]:
search("What does a development card cost?")

0.477  catan-p9-c0  |  'CITIES Cities are worth 2 VPs.\n\nBUILDING COST:\n\nCities always replace settlements. To build a city, '
0.457  catan-p8-c0  |  'BUILD\n\nRoads, buildings (settlements and cities), and development cards each have different costs. D'
0.415  catan-p9-c1  |  'Development cards never go back into the supply. If the supply runs out, you may not build any more '
0.293  catan-p11-c1  |  'Sort the resource cards by type and place them in five faceup stacks in the card trays. Shuffle the '
0.284  catan-p3-c1  |  'Settlement\n\nVP 1\n\nCity\n\nVPs 2\n\n• In any order (optional): • Trade • Build roads, settlements, cities'


In [7]:
import csv

def parse_sources(s):
    pairs = set()
    for piece in s.split(";"):                 # "catan.pdf:6;catan.pdf:9" → ["catan.pdf:6", "catan.pdf:9"]
        file, page = piece.strip().split(":")  # "catan.pdf:6" → "catan.pdf", "6"
        if page == "all":                      # unanswerable row: no right page exists
            return None
        game = file.replace(".pdf", "")        # "catan.pdf" → "catan"
        pairs.add((game, int(page)))           # "6" → 6, to match chunk["page"]
    return pairs

print(parse_sources("catan.pdf:6;catan.pdf:9"))   # {('catan', 6), ('catan', 9)}
print(parse_sources("catan.pdf:all"))             # None

{('catan', 9), ('catan', 6)}
None


In [8]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=5):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve(row["question"], k=5)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@5: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@5: 31/35 = 0.89

Misses:
('13', 'direct_lookup', 'In four-player Standard Rummy, how many cards is each player dealt?')
('18', 'table_figure', 'Using the Card/Value table, how many points does the winner receive from an opponent holding an Ace, a Jack, and a seven?')
('20', 'direct_lookup', 'What happens to your original bet on a push?')
('40', 'disambiguation', 'What is an Ace worth?')


In [9]:
def debug_miss(question_id):
    row = next(r for r in questions if r["id"] == str(question_id))
    expected = parse_sources(row["sources"])
    print("Q:", row["question"])
    print("Expected:", expected, "\n")

    q = model.encode([row["question"]], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    order = np.argsort(scores)[::-1]

    print("Top 5:")
    for i in order[:5]:
        print(f"  {scores[i]:.3f}  {chunks[i]['id']}")

    print("\nBest correct chunk:")
    for rank, i in enumerate(order, start=1):
        if (chunks[i]["game"], chunks[i]["page"]) in expected:
            print(f"  rank {rank}, score {scores[i]:.3f}, {chunks[i]['id']}")
            print(f"  {chunks[i]['text'][:200]!r}")
            break

In [10]:
debug_miss(13)

Q: In four-player Standard Rummy, how many cards is each player dealt?
Expected: {('rummy', 4)} 

Top 5:
  0.663  rummy-p6-c0
  0.625  poker-p6-c1
  0.597  rummy-p1-c2
  0.590  rummy-p1-c1
  0.581  rummy-p1-c0

Best correct chunk:
  rank 6, score 0.575, rummy-p4-c1
  'The Deal: The number of cards dealt depends on the number of players. If there are 2 players then they get 10 cards each. If there are 3 or 4 players then 7 cards, 5 or 6 players get 6 cards each.\n\nOb'


In [11]:
debug_miss(18)

Q: Using the Card/Value table, how many points does the winner receive from an opponent holding an Ace, a Jack, and a seven?
Expected: {('rummy', 5), ('rummy', 6)} 

Top 5:
  0.637  rummy-p2-c2
  0.561  blackjack-p2-c1
  0.533  rummy-p4-c1
  0.528  texas_holdem-p1-c2
  0.513  poker-p2-c1

Best correct chunk:
  rank 10, score 0.473, rummy-p6-c0
  '10\n\nFace cards\n\nOthers Face value\n\nThe total value of all remaining cards are added to the score of the player who won the hand. Additional hands are played until a predetermined cumulative score is r'


In [12]:
debug_miss(20)

Q: What happens to your original bet on a push?
Expected: {('blackjack', 2)} 

Top 5:
  0.547  texas_holdem-p3-c3
  0.544  texas_holdem-p3-c0
  0.525  poker-p3-c0
  0.523  texas_holdem-p3-c1
  0.516  texas_holdem-p3-c2

Best correct chunk:
  rank 8, score 0.486, blackjack-p2-c0
  "Dealer's Turn\n\nDealer reveals the hole card.\n\nDealer must hit until reaching 17 or higher.\n\nIf the dealer busts (goes over 21), all remaining players win.\n\nPAYOUTS\n\n● Win: Paid 1:1. ● Blackjack (Ace +"


In [13]:
debug_miss(40)

Q: What is an Ace worth?
Expected: {('blackjack', 1), ('rummy', 6), ('rummy', 4), ('poker', 2)} 

Top 5:
  0.639  rummy-p2-c2
  0.475  blackjack-p2-c1
  0.437  rummy-p2-c1
  0.409  texas_holdem-p1-c2
  0.371  poker-p4-c2

Best correct chunk:
  rank 7, score 0.309, blackjack-p1-c0
  'Blackjack is a fast, social casino classic where players compete only against the dealer, not each other. The goal is simple: build a hand closer to 21 than the dealer without busting. With quick deci'


id, question_type, cause, right_chunk's_rank

13, direct_lookup, right_game & wrong_page, 6

18, table_figure, right_game & wrong_page, 10

20, direct_lookup, wrong_game, 8

40, disambiguation, wrong game, 7


In [15]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=10):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve(row["question"], k=10)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@10: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@10: 35/35 = 1.00

Misses:


In [16]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_reranked(question, k=5, candidates=10):
    shortlist = retrieve(question, k=candidates)          # stage 1: fast, top 10
    pairs = [(question, c["text"]) for c in shortlist]    # each (question, chunk) side by side
    scores = reranker.predict(pairs)                      # stage 2: careful score for each pair
    order = np.argsort(scores)[::-1][:k]                  # best 5 by the new scores
    return [shortlist[i] for i in order]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16741.50it/s]


In [17]:
row = next(r for r in questions if r["id"] == "20")
for c in retrieve_reranked(row["question"]):
    print(c["id"])

blackjack-p2-c0
texas_holdem-p3-c0
blackjack-p1-c1
texas_holdem-p4-c0
texas_holdem-p3-c1


In [18]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=5):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve_reranked(row["question"], k=5)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@5: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@5: 35/35 = 1.00

Misses:


In [19]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=1):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve_reranked(row["question"], k=1)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@1: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@1: 26/35 = 0.74

Misses:
('3', 'myth_buster', 'Is every development card forbidden from being used on the turn you buy it?')
('16', 'disambiguation', "In Standard Rummy, how many points is an Ace left in an opponent's hand worth under the default rule?")
('18', 'table_figure', 'Using the Card/Value table, how many points does the winner receive from an opponent holding an Ace, a Jack, and a seven?')
('19', 'direct_lookup', 'At what hand total may the dealer stop taking cards?')
('21', 'multi_hop', 'Your initial two cards are an Ace and a Queen. What is their total, and what payout does the book list for that hand?')
('23', 'disambiguation', 'In Blackjack, is an Ace always worth one point, as it is by default in Standard Rummy?')
('27', 'multi_hop', 'A player checks, then faces a bet later in the round and folds. Must they call that bet, and do they recover chips already committed to the pot?')
('37', 'disambiguation', 'Can jokers be used in the deck?')
('40', 'disambiguation', '

In [20]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=3):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve_reranked(row["question"], k=3)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@3: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@3: 34/35 = 0.97

Misses:
('23', 'disambiguation', 'In Blackjack, is an Ace always worth one point, as it is by default in Standard Rummy?')


In [21]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=1):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve(row["question"], k=1)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@1: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@1: 23/35 = 0.66

Misses:
('6', 'table_figure', 'According to the pictured building-cost diagram, what does a development card cost?')
('12', 'table_figure', 'On the illustrated Pentonville Road Title Deed, what is the rent with three Houses?')
('13', 'direct_lookup', 'In four-player Standard Rummy, how many cards is each player dealt?')
('14', 'multi_hop', 'In five-player Standard Rummy, the player left of the dealer already holds a valid set. How many cards were they dealt, and may they meld before drawing?')
('16', 'disambiguation', "In Standard Rummy, how many points is an Ace left in an opponent's hand worth under the default rule?")
('18', 'table_figure', 'Using the Card/Value table, how many points does the winner receive from an opponent holding an Ace, a Jack, and a seven?')
('19', 'direct_lookup', 'At what hand total may the dealer stop taking cards?')
('20', 'direct_lookup', 'What happens to your original bet on a push?')
('21', 'multi_hop', 'Your initial two cards are

In [22]:
with open("../eval/questions.csv") as f:
    questions = list(csv.DictReader(f))    # each row becomes a dict: row["question"], row["sources"], ...

def retrieve(question, k=3):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]        # same as search(), but returns chunks instead of printing

hits = 0
scored = 0
misses = []

for row in questions:
    expected = parse_sources(row["sources"])
    if expected is None:                   # skip unanswerable rows
        continue
    scored += 1
    retrieved = retrieve(row["question"], k=3)
    found = any((c["game"], c["page"]) in expected for c in retrieved)
    if found:
        hits += 1
    else:
        misses.append((row["id"], row["question_type"], row["question"]))

print(f"Recall@3: {hits}/{scored} = {hits / scored:.2f}")
print("\nMisses:")
for m in misses:
    print(m)

Recall@3: 31/35 = 0.89

Misses:
('13', 'direct_lookup', 'In four-player Standard Rummy, how many cards is each player dealt?')
('18', 'table_figure', 'Using the Card/Value table, how many points does the winner receive from an opponent holding an Ace, a Jack, and a seven?')
('20', 'direct_lookup', 'What happens to your original bet on a push?')
('40', 'disambiguation', 'What is an Ace worth?')
